# Zuber 
## Chicago 2017
### Preferencias de los pasajeros y el impacto de los factores externos en los viajes

#### 1. Analizar los datos meteorológicos  

In [1]:
#Analizando datos del clima en Chicago en noviembre de 2017

#importar librerías
import requests
from bs4 import BeautifulSoup
import pandas as pd 

#solicitud
URL='https://practicum-content.s3.us-west-1.amazonaws.com/data-analyst-eng/moved_chicago_weather_2017.html'
req = requests.get(URL)
soup = BeautifulSoup(req.text, 'lxml')
table = soup.find('table', attrs={'id':'weather_records'})
#nombres de columnas
heading_table=[]
for row in table.find_all('th'):
    heading_table.append(row.text)
#contenido de la tabla
content=[]
for row in table.find_all('tr'):
    #que no sea el encabezado
    if not row.find_all('th'):
        content.append([element.text for element in row.find_all('td')])
#crear dataframe
weather_records = pd.DataFrame(content, columns=heading_table)
print(weather_records)

           Date and time Temperature       Description
0    2017-11-01 00:00:00     276.150     broken clouds
1    2017-11-01 01:00:00     275.700  scattered clouds
2    2017-11-01 02:00:00     275.610   overcast clouds
3    2017-11-01 03:00:00     275.350     broken clouds
4    2017-11-01 04:00:00     275.240     broken clouds
..                   ...         ...               ...
692  2017-11-29 20:00:00     281.340        few clouds
693  2017-11-29 21:00:00     281.690      sky is clear
694  2017-11-29 22:00:00     281.070        few clouds
695  2017-11-29 23:00:00     280.060      sky is clear
696  2017-11-30 00:00:00     278.460      sky is clear

[697 rows x 3 columns]


#### 2. Análisis exploratorio

1. CONSULTA SQL PARA OBTENER LA CANTIDAD DE VIAJES EN TAXI PARA CADA COMPAÑIAPARA EL 15 Y 16 DE NOVIEMBRE DE 2017 
SELECT 
    company_name,
    COUNT(trip_id) AS trips_amount
FROM 
    trips
    LEFT JOIN cabs ON trips.cab_id=cabs.cab_id
WHERE 
    start_ts::date BETWEEN '2017-11-15' AND '2017-11-16' 
GROUP BY 
    company_name
ORDER BY
trips_amount DESC

2. CONSULTA SQL PARA OBTENER CANTIDAD DE VIAJES PARA CADA EMPRESA SI EN SU NOMBRE TIENEN 'Blue' O 'Yellow' DEL 1 AL 7 DE NOVIEMBRE
SELECT
    company_name,
    COUNT(trip_id) AS trips_amount
FROM trips
    LEFT JOIN cabs ON trips.cab_id=cabs.cab_id
WHERE 
    (start_ts::date BETWEEN '2017-11-01' AND '2017-11-07') AND (company_name LIKE '%Blue%' OR company_name LIKE '%Yellow%')
GROUP BY company_name
ORDER BY 
    trips_amount

3. CONSULTA DE SQL PARA OBTENER EL NUMERO DE VIAJES DE LA EMPRESA FLASH CAB Y TAXI AFFILIATION SERVICES DEL 1 AL 7 DE NOVIEMBRE AGRUPANDO LO DEMÁS COMO 'OTHER'
SELECT 
    CASE 
        WHEN company_name IN ('Flash Cab','Taxi Affiliation Services') 
            THEN company_name
        ELSE 'Other'
    END AS company,    
    COUNT(trip_id) AS trips_amount
FROM trips
    INNER JOIN cabs ON trips.cab_id=cabs.cab_id
WHERE 
    start_ts::date BETWEEN '2017-11-01' AND '2017-11-07'
GROUP BY company
ORDER BY trips_amount DESC

#### 3. Pruebla de hipótesis sobre Loop y O'Hale

4. CONSULTA DE SQL PARA OBTENER IDENTIFICADORES DE LAS UBICACIONES. LOOP Y O'HALE. 
SELECT 
    neighborhood_id,
    name
FROM neighborhoods
WHERE name LIKE '%Hare' or name LIKE 'Loop'

5. CPNSULTA DE SQL PARA OBTENER PARA CADA HORA LOS REGISTROS METEREOLÓGICOS DIVIDIENDO EN BUENOS Y MALOS. CONDICIONES CLIMÁTICAS
SELECT 
    ts,
    CASE 
        WHEN description LIKE '%rain%' OR description LIKE '%storm%' THEN 'Bad'
    ELSE 'Good'
    END AS weather_conditions
FROM weather_records 


6. CONSULTA DE SQL PARA OBTENER LA DURACIÓN EN SEGUNDOS, LAS CONDICIONES CLIMÁTICAS (weather_conditions) DE TODOS LOS VIAJES QUE COMENZARON EN EL LOOP ID:50 EL SÁBADO Y TERMINARONN EN O'HARE ID:63 
SELECT
    trips.start_ts,
    CASE 
        WHEN description LIKE '%rain%' OR description LIKE '%storm%' THEN 'Bad'
    ELSE 'Good'
    END AS weather_conditions,
    duration_seconds
FROM trips
    INNER JOIN weather_records ON trips.start_ts=weather_records.ts 
WHERE 
    pickup_location_id = 50 AND dropoff_location_id = 63 AND EXTRACT(DOW FROM trips.start_ts)=6
ORDER BY
trips.trip_id